# Проект модуля. Нейросеть для автодополнения текстов

## Бизнес-контекст и задача проекта
Соцсетевое приложение, где пользователи постят короткие тексты. В продукте стоит задача — добавить возможность автодополнения текстов. Необходимо оздать модель, которую можно запускать на мобильных устройствах. Для смартфонов есть значительные требования по оперативной памяти и скорости работы, так что важна легковесность модели. 

## Формальная задача
Разработчики предоставили вам датасет с короткими текстовыми постами. 
### Поэтапное описание задачи:
- Взять датасет от разработчиков, очистить его, подготовить для обучения модели.
- Реализовать и обучить модель на основе рекуррентных нейронных сетей.
- Замерить качество разработанной и обученной модели.
- Взять более «тяжёлую» предобученную модель из Transformers и замерить её качество.
- Проанализировать результаты и дать рекомендации разработчикам: стоит ли использовать лёгкую модель или лучше постараться поработать с ограничениями по памяти и использовать большую предобученную.

## Загрузка библиотек

In [44]:
#Автоматическая перезагрузка при изменениях
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
# Для MPS рекомендуется включить fallback на CPU для неподдерживаемых операций
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [45]:
from src.data_utils import load_and_clear_data, load_or_tokenize, create_sequences, create_data_split, build_vocab
from src.constants import BATCH_SIZE
from src.next_token_dataset import NextTokenDataset
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_model
from src.lstm_test import test_model
from src.lstm_utils import save_model_weight
from src.eval_transformer_pipeline import evaluate_transformer

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

## Этап 1. Сбор и подготовка данных

In [4]:
texts = load_and_clear_data()

Загружено 10000 текстов, после очистки осталось 9979 текстов.
Примеры очищенных текстов: ['a thats a bummer. you shoulda got david carr of third day to do it. d', 'is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!', 'i dived many times for the ball. managed to save 50 the rest go out of bounds', 'my whole body feels itchy and like its on fire', 'no, its not behaving at all. im mad. why am i here? because i cant see you all over there.']


In [5]:
tokenized_texts = load_or_tokenize(texts)

Загрузка токенизированных данных из data/tokenized_texts.pkl...


In [6]:
X, Y = create_sequences(tokenized_texts)

In [7]:
display(X[1:2])
display(Y[1:2])

[['a',
  'thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd']]

[['thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd',
  '<EOS>']]

In [8]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = create_data_split(X, Y) 

Train: 127046, Val: 15881, Test: 15881


In [9]:
lengths = [len(tokens) for tokens in tokenized_texts]
print(f"Средняя длина: {sum(lengths) / len(lengths):.1f}")
print(f"Медиана: {sorted(lengths)[len(lengths)//2]}")
print(f"90-й перцентиль: {sorted(lengths)[int(len(lengths)*0.9)]}")

Средняя длина: 16.9
Медиана: 16
90-й перцентиль: 28


In [10]:
# Создание словаря
word2idx, idx2word = build_vocab(tokenized_texts)

vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 13784


In [11]:
# Создание датасетов
train_dataset = NextTokenDataset(X_train, Y_train, word2idx)
val_dataset = NextTokenDataset(X_val, Y_val, word2idx)
test_dataset = NextTokenDataset(X_test, Y_test, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

## Этап 2. Объявление модели LSTM

In [12]:
device = (
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else
        "cpu"
    )
print(f"Using device: {device}")

print("torch.backends.mps.is_available:", torch.backends.mps.is_available())
print("torch.backends.mps.is_built:", torch.backends.mps.is_built())

Using device: mps
torch.backends.mps.is_available: True
torch.backends.mps.is_built: True


In [49]:
model = LSTMLanguageModel(vocab_size, word2idx, idx2word).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Этап 3. Тренировка модели LSTM

In [51]:
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, idx2word, word2idx)

Epoch 1/10 [Train]: 100%|██████████| 993/993 [01:00<00:00, 16.44it/s, loss=5.2399]



🥑Примеры генерации модели (LSTM)

[Пример 1]
Вход (75%):     waiting at the airport for my ride while i get harassed by 2 men trying to sell me ugly hats .. why me
Цель (25%):     ? ! i just want to sleep ..
Предсказание:   start of the but my completely sick though
Полное предсказание:   waiting at the airport for my ride while i get harassed by 2 men trying to sell me ugly hats .. why me start of the but my completely sick though

[Пример 2]
Вход (75%):     so do not want to be awake right now
Цель (25%):     boo school !
Предсказание:   
Полное предсказание:   so do not want to be awake right now

[Пример 3]
Вход (75%):     her mama then she is ....
Цель (25%):     hahaha lame !
Предсказание:   bad .
Полное предсказание:   her mama then she is .... bad .

[Пример 4]
Вход (75%):     my birthday next
Цель (25%):     week woot
Предсказание:   snowing ,
Полное предсказание:   my birthday next snowing ,

[Пример 5]
Вход (75%):     baking oatmeal chocolate chip cookies to make me tired
Ц

Epoch 2/10 [Train]:  42%|████▏     | 418/993 [00:25<00:34, 16.54it/s, loss=5.3534]


KeyboardInterrupt: 

In [23]:
save_model_weight(model)

✅ Веса модели сохранены в models/lstm_model_weights.pth


In [17]:
test_model(model, test_loader, idx2word, word2idx, device, criterion)


ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ (10%)
Test Loss: 1.9903
Test ROUGE-1: 0.6969, Test ROUGE-2: 0.5911 


### Этап 4. Использование предобученного трансформера

In [37]:
evaluate_transformer(val_loader, idx2word, device=device)


Оценка на валидационной выборке (DISTILGPT2)


Processing batches:   0%|          | 0/125 [00:00<?, ?it/s]


[Пример 1]
Вход (75%):     <BOS> waiting at the airport for my ride while i get harassed by 2 men trying to sell me ugly hats .. why me
Цель (25%):     ? ! i just want to sleep ..
Предсказание:   and my friends can't be at the airport without the police ?


@brosha

[Пример 2]
Вход (75%):     so do not want to be awake right now
Цель (25%):     boo school !
Предсказание:   and not want to wake up. The answer is, I don't like to wake up, I

[Пример 3]
Вход (75%):     her mama then she is ....
Цель (25%):     hahaha lame !
Предсказание:   . She has an amazing personality, I think she is very strong at heart. She has a lot

[Пример 4]
Вход (75%):     my birthday next
Цель (25%):     week woot
Предсказание:   year, I›m going to go and get the rest of my friends and family for the

[Пример 5]
Вход (75%):     baking oatmeal chocolate chip cookies to make me tired
Цель (25%):     . i cant sleep
Предсказание:   .
I'm really into chocolate but it's my first time ever doing so.
I like

[Пример 

Processing batches:   8%|▊         | 10/125 [03:56<45:18, 23.64s/it] 


KeyboardInterrupt: 